In [29]:
!pip install chardet
#!pip install pandas
#!pip install --upgrade pip

In [30]:
import pandas as pd 
import os
import chardet
import glob

In [31]:
os.getcwd()

'/home/rstudio/data-coration-eLTER/eLTER-data-download/elter-data-curator-tools'

In [49]:
# Import dataset by chanching the "dataset_dir" file for each parent folder and chanching BASE_DIR to your curator-tool folder

from pathlib import Path
import re

base = "/home/rstudio/"
BASE_DIR = Path(f"{base}data-coration-eLTER/eLTER-data-download/elter-data-curator-tools")
dataset_dir = Path(f"{base}data-coration-eLTER/eLTER-data-download/elter-data-curator-tools/downloads_IDs/IT25_-_Val_Mazia-Matschertal_-_Italy/33130-yn348") #diesen Pfad immer änderen 

# case-insensitive
patterns = {
    "data": r"data",
    "method": r"method",
    "station": r"station",
    "reference": r"reference",
    "license": r"licen[sc]e",  # both "license" and "licence"
}

filepaths = {}

for key, pattern in patterns.items():
    matches = [
        str(f) for f in dataset_dir.iterdir()
        if f.is_file() and re.search(pattern, f.name, re.IGNORECASE)
    ]
    if key == "data":
        filepaths[key] = sorted(matches)  # more than one data file
    else:
        filepaths[key] = matches[0] if matches else None 


In [50]:
# Basic information from DAR for the report 2.

import requests

def load_token(base_dir):
    token_file = Path(base_dir) / ".elter-dar-access-token"
    with open(token_file) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                return line
    raise ValueError("no Token in .elter-dar-access-token gefunden")

def get_dataset_info(dataset_id, base_dir):
    token = load_token(base_dir)
    url = f"https://dar.elter-ri.eu/api/datasets/{dataset_id}/draft"
    resp = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    resp.raise_for_status()
    full = resp.json()
    meta = full.get("metadata", full)  # falls "metadata" fehlt, nimm das Top-Level-JSON

    info = {
        "Title": meta.get("titles", [{}])[0].get("titleText") if meta.get("titles") else None,
        "Organizations": [o.get("organizationName") for o in meta.get("responsibleOrganizations", [])],
        "Creators": [
            f"{c.get('creatorGivenName','')} {c.get('creatorFamilyName','')}".strip()
            for c in meta.get("creators", [])
        ],
        "Creator emails": [c.get("creatorEmail") for c in meta.get("creators", [])],
        "Dataset Type": meta.get("datasetType", {}).get("datasetTypeCode"),
        "Temporal Coverages": [
            f"{tc.get('startDate')} to {tc.get('endDate')}"
            for tc in meta.get("temporalCoverages", [])
        ],
        "Standard observation habitats": [
            h.get("soHabitatCode") for h in meta.get("habitatReferences", [])
        ],
        "Site name": meta.get("siteReferences", [{}])[0].get("siteName") if meta.get("siteReferences") else None,
    }
    return info, full

dataset_id = Path(dataset_dir).name
print(f"Dataset-ID: {dataset_id}")

info, raw = get_dataset_info(dataset_id, BASE_DIR)
info


Dataset-ID: 33130-yn348


{'Title': 'Mazia-Matschertal, 2014-2017, soil water potential (swp), soil temperature (ts), and volume water content (vwc)',
 'Organizations': ['Eurac Research'],
 'Creators': ['Alessandro Zandonai'],
 'Creator emails': ['Alessandro.Zandonai@eurac.edu'],
 'Dataset Type': 'SOGEO_001',
 'Temporal Coverages': ['2014-04-09 to 2017-12-31'],
 'Standard observation habitats': ['I'],
 'Site name': 'IT25 - Val Mazia-Matschertal - Italy'}

In [55]:
# Preparing data for analysis: check the "encoding", "separator", and "skip-row" settings.

import csv

def detect_encoding(filepath, sample_size=10000):
    with open(filepath, "rb") as f:
        raw = f.read(sample_size)

    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    if raw.startswith(b"\xff\xfe") or raw.startswith(b"\xfe\xff"):
        return "utf-16"

    try:
        result = from_path(filepath).best()
        if result is not None and result.encoding:
            with open(filepath, encoding=result.encoding) as f:
                f.read(sample_size)
            return result.encoding
    except (UnicodeDecodeError, LookupError, NameError):
        pass

    candidates = ["utf-8", "cp1252", "iso-8859-1", "iso-8859-15", "latin-1"]
    for enc in candidates:
        try:
            with open(filepath, encoding=enc) as f:
                f.read(sample_size)
            return enc
        except (UnicodeDecodeError, LookupError):
            continue

    return "latin-1"

def detect_sep(filepath, encoding):
    with open(filepath, encoding=encoding, errors="ignore") as f:
        sample = f.read(4096)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        return dialect.delimiter
    except csv.Error:
        return ","

def detect_skiprows(filepath, encoding, sep, max_check=20):
    with open(filepath, encoding=encoding, errors="ignore") as f:
        lines = [next(f, "") for _ in range(max_check)]

    for i, line in enumerate(lines):
        fields = line.strip().split(sep)
        if len(fields) > 1:
            for next_line in lines[i+1:]:
                if next_line.strip():
                    next_fields = next_line.strip().split(sep)
                    if abs(len(next_fields) - len(fields)) <= 1:
                        return i
                    break
    return 0

def smart_read_csv(filepath, **kwargs):
    filepath = str(filepath)
    encoding = detect_encoding(filepath)
    sep = detect_sep(filepath, encoding)
    skiprows = detect_skiprows(filepath, encoding, sep)

    df = pd.read_csv(filepath, sep=sep, skiprows=skiprows, encoding=encoding, **kwargs)
    df = df.dropna(axis=1, how="all")
    df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]

    print(f"{Path(filepath).name}: encoding={encoding}, sep={repr(sep)}, skiprows={skiprows}")
    return df

In [56]:
# read ant plot the data

#df_data = pd.read_csv(filepaths["data"], sep=",", low_memory=False, skiprows=10) #, encoding="latin-1",
#df_data.head() 
#if mor than one data-file
#df_data = pd.concat([pd.read_csv(p,sep=",") for p in filepaths["data"]], ignore_index=True)#skiprows=11
#pd.set_option('display.max_rows', None) 
df_data = pd.concat([smart_read_csv(p) for p in filepaths["data"]], ignore_index=True)
df_data.head(10)
#df_data.iloc[20:30]


IT_ValMazia-Matschertal_SOIL_swp_20_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_swp_5_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_ts_20_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_ts_2_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_ts_5_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_vwc_20_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_vwc_2_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0
IT_ValMazia-Matschertal_SOIL_vwc_5_2014-2017_V20251027_data.csv: encoding=utf-8, sep=',', skiprows=0


,SITE_CODE,STATION_CODE,VARIABLE,LEVEL,TIME,VALUE,FLAGQUA
0,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T11:45+01:00,-37.59,OK
1,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T12:00+01:00,-37.55,OK
2,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T12:15+01:00,-37.55,OK
3,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T12:30+01:00,-37.53,OK
4,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T12:45+01:00,-37.53,OK
5,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T13:00+01:00,-37.50,OK
6,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T13:15+01:00,-37.50,OK
7,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T13:30+01:00,-37.47,OK
8,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T13:45+01:00,-37.46,OK
9,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,swp,-20,2016-04-21T14:00+01:00,-37.45,OK


In [59]:
# show method and check if every 

df_method = smart_read_csv(filepaths["method"])
df_method.head(20)

target_cols = {"variable": None, "meth_descr": None}
for col in df_method.columns:
    col_lower = col.strip().lower()
    if col_lower in target_cols:
        target_cols[col_lower] = col  # keep the actual, originally written column name

print("Found columns:", target_cols)

found_cols = [v for v in target_cols.values() if v is not None]

if target_cols["variable"] is not None and target_cols["meth_descr"] is not None:
    print("✓ Both 'Variable' and 'Meth_descr' are present — the method file is complete.")
    display(df_method[found_cols].head(20))
elif found_cols:
    missing = [k for k, v in target_cols.items() if v is None]
    print(f"⚠ Incomplete: missing column(s) {missing} — method file is NOT complete.")
    display(df_method[found_cols].head(20))
else:
    print("✗ Neither 'Variable' nor 'Meth_descr' found (in any spelling) — method file is incomplete.")

IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_method.csv: encoding=utf-8-sig, sep=',', skiprows=0
Found columns: {'variable': 'VARIABLE', 'meth_descr': 'METH_DESCR'}
✓ Both 'Variable' and 'Meth_descr' are present — the method file is complete.


,VARIABLE,METH_DESCR
0,vwc_2,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
1,vwc_5,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
2,vwc_20,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
3,ts_2,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
4,ts_5,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
5,ts_20,Sensor Type: Soil water content reflectometer - Sensor Model: Campbell Scientific CS655 - Automatic monitoring station samples data every minute and aggregates every 15 minutes
6,swp_5,"Sensor Type: Dielectric water potential sensors - Sensor Model: - Decagon MPS-6, Meter Group Teros 21 - Automatic monitoring station samples data every minute and aggregates every 15 minutes or stores the last one every 15 minutes"
7,swp_20,"Sensor Type: Dielectric water potential sensors - Sensor Model: - Decagon MPS-6, Meter Group Teros 21 - Automatic monitoring station samples data every minute and aggregates every 15 minutes or stores the last one every 15 minutes"


In [60]:
# show station file and check if it is complete

df_station = smart_read_csv(filepaths["station"])
df_station

required_station_cols = {
    "site_code": None,
    "station_code": None,
    "stype": None,
    "lat": None,
    "lon": None,
    "altitude": None,
}

for col in df_station.columns:
    col_lower = col.strip().lower()
    if col_lower in required_station_cols:
        required_station_cols[col_lower] = col  # keep the actual, originally written column name

print("Found columns:", required_station_cols)

found_cols = [v for v in required_station_cols.values() if v is not None]
missing = [k for k, v in required_station_cols.items() if v is None]

if not missing:
    print("✓ All required station columns are present — the station file is complete.")
    display(df_station[found_cols].head(20))
elif found_cols:
    print(f"⚠ Incomplete: missing column(s) {missing} — station file is NOT complete.")
    display(df_station[found_cols].head(20))
else:
    print("✗ None of the required station columns found — station file is incomplete.")

IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_station.csv: encoding=utf-8-sig, sep=',', skiprows=0
Found columns: {'site_code': 'SITE_CODE', 'station_code': 'STATION_CODE', 'stype': 'STYPE', 'lat': 'LAT', 'lon': 'LON', 'altitude': 'ALTITUDE'}
✓ All required station columns are present — the station file is complete.


,SITE_CODE,STATION_CODE,STYPE,LAT,LON,ALTITUDE
0,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b1,AREA,10.590244,46.661183,983
1,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b2,AREA,10.579917,46.686253,1473
2,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,p2,AREA,10.585125,46.684306,1541
3,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,b3,AREA,10.591936,46.691694,1922
4,https://deims.org/51d0598a-e9e1-4252-8850-60fc8f329aab,s4,AREA,10.602181,46.706981,2404
5,https://deims.org/80c56aed-48bc-4d00-9ac0-033effeab9d2,s3,AREA,10.710939,46.766786,2705


In [13]:
# if "license" is there
with open(filepaths["license"], "r") as f:
    print(f.read())

CC-BY-4.0-INT


In [29]:
# if "reference" is there
df_reference =  pd.read_csv(filepaths["reference"])
display(df_reference)

KeyError: 'reference'

In [71]:
# Output usefull for 3. Submission overview

# file names
print("-" * 55)
print("FILE INFO")
print("-" * 55)

pfad = dataset_dir

# parent folders
print(f"parent folders: {os.path.basename(pfad)}")
# subfiles
print("\nSubfiles:")
for datei in os.listdir(pfad):
    print(f"  - {datei}")

# ------------------------------------------------------------------

# data volume
print("-" * 55)
print("data volume")
print("-" * 55)

total = 0
for f in os.listdir(pfad):
    fp = os.path.join(pfad, f)
    if os.path.isfile(fp):
        total += os.path.getsize(fp)

print(f"File size: {total / (1024 * 1024):.5f} MB")

# ------------------------------------------------------------------
def find_column(df, candidates):
    """Searches for a column case-insensitively based on a list of possible names."""
    col_map = {c.strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in col_map:
            return col_map[candidate]
    return None

TIME_CANDIDATES = ["time", "timestamp", "datetime", "date_time", "date"]
VARIABLE_CANDIDATES = ["variable", "var", "parameter", "param"]
VALUE_CANDIDATES = ["value", "val", "measurement", "obs", "observation"]

time_col = find_column(df_data, TIME_CANDIDATES)
variable_col = find_column(df_data, VARIABLE_CANDIDATES)
value_col = find_column(df_data, VALUE_CANDIDATES)
is_long = variable_col is not None

print(f"Detected columns -> time: {time_col}, variable: {variable_col}, value: {value_col}")

# ------------------------------------------------------------------
print("\n" + "-" * 55)
print("STRUCTURE")
print("-" * 55)
print(f"Columns:     {df_data.columns.tolist()}")
print(f"Num columns: {len(df_data.columns)}")
print(f"Rows:        {len(df_data)}")
print(f"Format:      {'Long' if is_long else 'Wide'}")

# ------------------------------------------------------------------
print("\n" + "-" * 55)
print("TEMPORAL COVERAGE & RESOLUTION")
print("-" * 55)

def format_interval(td):
    if pd.isna(td):
        return "n/a"
    total_seconds = td.total_seconds()
    if total_seconds < 60:
        return f"{round(total_seconds)} seconds"
    elif total_seconds < 3600:
        return f"{round(total_seconds / 60)} minutes"
    elif total_seconds < 86400:
        return f"{round(total_seconds / 3600)} hours"
    elif total_seconds < 86400 * 28:
        return f"{round(total_seconds / 86400)} days"
    elif total_seconds < 86400 * 365:
        return f"{round(total_seconds / (86400 * 30.44))} months"
    else:
        return f"{round(total_seconds / (86400 * 365.25))} years"

def summarize_times(times):
    """Takes a series of timestamps (for one variable) and computes coverage + resolution."""
    times = pd.to_datetime(times, errors='coerce').dropna().sort_values().drop_duplicates()
    n = len(times)
    if n == 0:
        return pd.Series({'n': 0, 'From': "n/a", 'To': "n/a", 'Resolution': "n/a",
                           'Regularity': "n/a", 'Min interval': "n/a", 'Max interval': "n/a"})

    from_ = times.min().strftime('%Y-%m-%d')
    to_ = times.max().strftime('%Y-%m-%d')

    diffs = times.diff().dropna()
    if diffs.empty:
        return pd.Series({'n': n, 'From': from_, 'To': to_, 'Resolution': "only 1 timestamp",
                           'Regularity': "n/a", 'Min interval': "n/a", 'Max interval': "n/a"})

    median = diffs.median()
    tolerance = pd.Timedelta(minutes=1)
    regular_share = (diffs - median).abs() <= tolerance
    if regular_share.all():
        regularity = "regularly"
    elif regular_share.mean() >= 0.99:
        regularity = "mostly regularly"
    else:
        regularity = "irregularly"

    return pd.Series({
        'n': n, 'From': from_, 'To': to_,
        'Resolution': format_interval(median),
        'Regularity': regularity,
        'Min interval': format_interval(diffs.min()),
        'Max interval': format_interval(diffs.max()),
    })

if time_col is None:
    print("✗ No recognizable time column found (checked: " + ", ".join(TIME_CANDIDATES) + ")")
else:
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    if is_long:
        # LONG FORMAT: gruppieren nach variable_col
        subset_cols = [c for c in [time_col, value_col] if c is not None]
        df_valid = df_data.dropna(subset=subset_cols)

        summary = df_valid.groupby(variable_col, group_keys=False).apply(
            lambda g: summarize_times(g[time_col]), include_groups=False
        )
        print(summary.to_string())

    else:
        # WIDE FORMAT: jede Nicht-Zeit-Spalte = eigene Variable
        candidate_value_cols = [c for c in df_data.columns if c != time_col]
        results = {}
        for col in candidate_value_cols:
            df_valid = df_data.dropna(subset=[time_col, col])
            results[col] = summarize_times(df_valid[time_col])

        summary = pd.DataFrame(results).T
        print(summary.to_string())

# ------------------------------------------------------------------

#File encoding 06.
print("\n" + "-" * 55)
print("File encoding 06")
print("-" * 55)

for name, paths in filepaths.items():
    if paths is None:
        print(f"{name}: ✗ no file found")
        continue

    if isinstance(paths, str):
        paths = [paths]

    for path in paths:
        try:
            with open(path, "rb") as f:
                raw = f.read(20000)
            result = chardet.detect(raw)
            encoding = result.get("encoding")

            if encoding is None:
                print(f"{Path(path).name}: ✗ encoding could not be determined")
            else:
                print(f"{Path(path).name}: {encoding}")

        except Exception as e:
            print(f"{Path(path).name}: ✗ error reading file — {e}")


-------------------------------------------------------
FILE INFO
-------------------------------------------------------
parent folders: 33130-yn348

Subfiles:
  - IT_ValMazia-Matschertal_SOIL_vwc_20_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_swp_5_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_ts_5_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_license.txt
  - IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_station.csv
  - IT_ValMazia-Matschertal_SOIL_ts_2_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_swp_20_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_method.csv
  - IT_ValMazia-Matschertal_SOIL_ts_20_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_2014-2017_V20251027_reference.csv
  - IT_ValMazia-Matschertal_SOIL_vwc_2_2014-2017_V20251027_data.csv
  - IT_ValMazia-Matschertal_SOIL_vwc_5_2014-2017_V20251027_data.csv
----------------------------------